# 12C: Do Waymo Vehicles Crash More Often in Some Cities Than Others?

## Background: Self-Driving Cars Are on Real Streets

Waymo is one of the most advanced autonomous vehicle (self-driving car) companies in the world. Unlike a human driver, a Waymo vehicle uses cameras, radar, and AI to navigate city streets — without a person touching the steering wheel.

Since 2020, Waymo has operated **robotaxi services** in multiple U.S. cities, including San Francisco, Phoenix, Los Angeles, Austin, and Atlanta. When one of their vehicles is involved in a crash, Waymo is required to file a public safety report. Waymo publishes **all** of these reports on their website — more transparency than most human drivers ever face!

You can explore their data here: https://waymo.com/safety/impact/

<img alt="A Waymo self-driving taxi" src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/Waymo_Driver_-_The_Waymo_One_Journey_Continues.jpg/1280px-Waymo_Driver_-_The_Waymo_One_Journey_Continues.jpg" title="A Waymo self-driving taxi" width=500>

*Image: A Waymo One robotaxi. (CC BY-SA 4.0, Wikimedia Commons)*

### The Question for Today

San Francisco is dense and hilly, with narrow streets, cyclists, pedestrians, and cable cars. Phoenix is flat and sprawling, with wide roads and less foot traffic.

**Today's investigation question:** *In an average month, does Waymo get into more crashes in San Francisco than in Phoenix — and how confident can we be in that estimate?*

In [ ]:
# Load the CourseKata library
suppressPackageStartupMessages({
    library(coursekata)
})

# What emoji will this be?
print("\U0001F697")

## 1.0 - Getting the Data

The `WaymoCrashesMonthly` data frame below has **one row per city per month**, covering September 2020 through September 2025. The crash counts come from Waymo's public safety reports.

Here are the variables we'll use today:

- `Location` — The city (`SAN_FRANCISCO`, `PHOENIX`, `LOS_ANGELES`, `AUSTIN`, `ATLANTA`)
- `Year_Month` — The month (e.g., `202309` = September 2023)
- `Crashes` — How many Waymo crashes were reported **that month in that city**
- `Injuries` — How many of those crashes involved at least one injury
- `Police_Reports` — How many were reported to police
- `Crash_IPMM` — Crashes per million miles driven (a rate that accounts for how much each city drives)

In [ ]:
# Load the pre-wrangled monthly dataset
WAYMO_URL <- "YOUR_GOOGLE_SHEETS_PUB_CSV_URL_HERE"

WaymoCrashesMonthly <- read.csv(WAYMO_URL, header = TRUE) %>%
    mutate(
        Location = factor(Location),
        Date     = as.Date(Date)
    )

# Keep only the two cities with the most data for today
SF_Phoenix <- filter(WaymoCrashesMonthly, Location %in% c("SAN_FRANCISCO", "PHOENIX"))
SF_Phoenix$Location <- droplevels(SF_Phoenix$Location)

1.1, Discussion - This dataset has one row per city per month instead of one row per crash. What did we have to *do* to the original crash records to get here? What information did we gain — and what did we lose — by aggregating this way?
  
  

1.2 — Take a look at the data. How many months of data do we have for each city? Does the number surprise you?

In [ ]:
head(SF_Phoenix)
tally(~ Location, data = SF_Phoenix)

## 2.0 - Explore Variation in Monthly Crashes

2.1 — Before looking at the data, write your hypothesis as a word equation. Which city do you predict has more crashes per month, and why?
  
  

2.2 — Let's look at the distribution of monthly crashes in each city. What do you notice?

In [ ]:
gf_histogram(~ Crashes, data = SF_Phoenix, bins = 12) %>%
    gf_facet_grid(Location ~ .) %>%
    gf_labs(
        x     = "Crashes per month",
        title = "Distribution of monthly Waymo crashes: SF vs. Phoenix"
    )

2.3 — Compute the mean monthly crashes for each city. Is the difference what you predicted?

In [ ]:
mean(Crashes ~ Location, data = SF_Phoenix)

2.4, Discussion - San Francisco has more total crashes than Phoenix. But Waymo also drives *more miles* in San Francisco. Does the raw crash count fairly represent how dangerous each city is? What would make a fairer comparison?
  
  

2.5 — **(Optional)** Look at `Crash_IPMM` (crashes per million miles) instead of raw `Crashes`. Does the city difference look the same, bigger, or smaller when we account for how much driving happened?

In [ ]:
mean(Crash_IPMM ~ Location, data = SF_Phoenix)

## 3.0 - Modeling the City Effect

We want to model whether `Location` predicts `Crashes`. In GLM notation:

$$Crashes_i = b_0 + b_1 \cdot Location_i + e_i$$

Where `Location` is coded as **0 = PHOENIX** and **1 = SAN\_FRANCISCO**.

- $b_0$ = predicted average monthly crashes in Phoenix  
- $b_1$ = how many *more* (or fewer) crashes per month San Francisco has compared to Phoenix

3.1 — Fit the model and save it as `location_model`. What are $b_0$ and $b_1$?

In [ ]:
location_model <- lm(Crashes ~ Location, data = SF_Phoenix)
location_model

3.2 — Interpret $b_0$ and $b_1$ in plain English. Complete these sentences:

*"In Phoenix, Waymo averages about ____ crashes per month. San Francisco averages about ____ more/fewer crashes per month than Phoenix."*
  
  

3.3 — Run the ANOVA table. How much of the variation in monthly crashes does city explain (PRE)?

In [ ]:
supernova(location_model)

3.4, Discussion - Even if the difference between cities is real, month-to-month crash counts vary a lot within each city. What factors might cause some months to have more crashes than others, regardless of city?
  
  

## 4.0 - A DGP Where City Doesn't Matter

Our sample shows a difference between cities — but this is just one five-year window of data. Even if city had **zero** effect on crash counts in the real data-generating process ($\beta_1 = 0$), random chance could still produce a non-zero $b_1$ in our sample.

We can simulate this "city doesn't matter" DGP by `shuffle()`-ing the city labels, which breaks any real city–crash relationship.

4.1 — If $\beta_1 = 0$ in the real DGP, which function simulates this?

```r
do(1000) * b1(shuffle(Crashes) ~ Location, data = SF_Phoenix)

do(1000) * b1(Crashes ~ Location, data = resample(SF_Phoenix))
```
  
  

4.2 — Generate the sampling distribution of $b_1$ from the no-city-effect DGP, then visualize it. Mark our sample $b_1$ in green.

In [ ]:
sdob1_no_effect <- do(1000) * b1(shuffle(Crashes) ~ Location, data = SF_Phoenix)

sample_b1 <- coef(location_model)["LocationSAN_FRANCISCO"]

gf_histogram(~ b1, data = sdob1_no_effect,
             fill = ~ middle(b1, .95)) %>%
    gf_refine(scale_fill_manual(values = c("coral", "dodgerblue"))) %>%
    gf_vline(xintercept = sample_b1, color = "green4", linewidth = 1) %>%
    gf_labs(
        x     = "b1 (shuffled)",
        title = "Is our sample b1 (green) unusual under the no-city-effect DGP?"
    )

4.3 — Is our sample $b_1$ in the coral tails or the blue middle? What does the p-value from the ANOVA table (section 3.3) say?
  
  

## 5.0 - A DGP Where the City Effect Matches Our Sample

We've seen whether the null DGP is plausible. Now let's ask the opposite: what if the *real* city effect in the DGP is exactly like what we observed? Using `resample()`, we can simulate what other samples would look like from that kind of DGP.

5.1 — Generate the sampling distribution from a DGP that matches our sample.

In [ ]:
sdob1_same_effect <- do(1000) * b1(Crashes ~ Location, data = resample(SF_Phoenix))

gf_histogram(~ b1, data = sdob1_same_effect,
             fill = ~ middle(b1, .95)) %>%
    gf_refine(scale_fill_manual(values = c("coral", "turquoise"))) %>%
    gf_vline(xintercept = sample_b1, color = "green4", linewidth = 1) %>%
    gf_labs(
        x     = "b1 (resampled)",
        title = "Sampling distribution of b1 when DGP matches our sample"
    )

5.2 — Compare the two distributions. How are they similar? How are they different? Where is each one centered?
  
  

5.3, Discussion - Could the true $\beta_1$ be 0? Could it be our sample $b_1$? Could it be some other value entirely? From these two sampling distributions, what range of values for $\beta_1$ seems *plausible*?
  
  

## 6.0 - Confidence Intervals

A **95% confidence interval** gives us a range of plausible values for the true $\beta_1$. If we repeated this study many times, 95% of the confidence intervals we construct would contain the true $\beta_1$.

6.1 — Use `confint()` to find the 95% confidence interval for $\beta_1$.

In [ ]:
confint(location_model)

6.2 — Interpret the confidence interval in plain English. Fill in the blanks:

*"We are 95% confident that the true difference in average monthly crashes between San Francisco and Phoenix is somewhere between ____ and ____ crashes per month. In other words, in an average month, San Francisco has somewhere between ____ and ____ [more/fewer] crashes than Phoenix."*
  
  

6.3 — Does the confidence interval include 0? What does it mean if 0 is *inside* the interval? What does it mean if 0 is *outside* the interval?
  
  

6.4 — You can also extract the confidence interval from the resampled sampling distribution using the middle 95%. How does it compare to `confint()`?

In [ ]:
quantile(~ b1, data = sdob1_same_effect, probs = c(0.025, 0.975))

6.5, Discussion - `confint()` uses a mathematical formula (the t-distribution), while the quantile approach uses simulation. They give similar answers. Why do you think that is?
  
  

## 7.0 - What Does This Mean for Self-Driving Car Safety?

7.1 — Write a one-sentence conclusion that uses your confidence interval. Be specific: don't just say "SF has more crashes" — say *how many more* and *how sure* you are.
  
  

7.2, Discussion - We found that SF and Phoenix differ in monthly crash counts. But Waymo drives *more miles* in SF. Does the number of crashes per month fairly represent safety? What would you want to know to make a fairer comparison between cities?
  
  

7.3 — **(Extension)** Repeat the analysis using `Crash_IPMM` (crashes per million miles) as the outcome instead of raw `Crashes`. This controls for how much Waymo drives in each city. Does the confidence interval change? What does that tell you?
  
  

In [ ]:
# Extension: model crash rate instead of raw count
ipmm_model <- lm(Crash_IPMM ~ Location, data = SF_Phoenix)
ipmm_model
confint(ipmm_model)

7.4, Final discussion - Waymo argues that self-driving cars are safer than human-driven cars. Based on what you've learned today about confidence intervals and statistical reasoning, what would you need to know to evaluate that claim? What data, what model, and what kind of comparison would you want to see?